# Adult Census Income — a mid-size mixed-type benchmark

Case: [Adult Census Income](https://www.kaggle.com/datasets/uciml/adult-census-income)
(the classic UCI dataset: 32,561 rows, 14 mixed features, predict whether
income exceeds $50K/yr; **roc_auc**).

No live leaderboard here — instead this is the dataset every tabular method
has been benchmarked on for ~30 years, so published results make a stable
reference. This notebook also shows **presets**: `preset="balanced"` adds
gated ensembling on top of the defaults — a blend ships only if it is
statistically better than the single winner.

In [1]:
import json
import logging
import os
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

from honestml import AutoML, CVConfig, render_report, save_run_report

SEED = 42
DATA = Path("data/adult")
RESULTS = Path("results/adult")
RESULTS.mkdir(parents=True, exist_ok=True)
KAGGLE = os.environ.get("KAGGLE_BIN") or shutil.which("kaggle") or str(Path.home() / ".local" / "bin" / "kaggle")

logging.basicConfig(format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("honestml").setLevel(logging.INFO)

In [2]:
if not (DATA / "adult.csv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [KAGGLE, "datasets", "download", "-d", "uciml/adult-census-income", "-p", str(DATA), "--unzip"],
        check=True,
    )

In [3]:
df = pd.read_csv(DATA / "adult.csv").replace("?", pd.NA)
y = (df.pop("income") == ">50K").astype(int)
X = df
print(f"rows: {len(X)}, positive rate: {y.mean():.3f}")
X.head()

rows: 32561, positive rate: 0.241


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,90,<NA>,77053,HS-grad,9,Widowed,<NA>,Not-in-family,White,Female,0,4356,40,United-States
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States
2,66,<NA>,186061,Some-college,10,Widowed,<NA>,Unmarried,Black,Female,0,4356,40,United-States
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States


## Fit

In [4]:
model = AutoML(
    task="binary",
    metric="roc_auc",
    preset="balanced",
    cv=CVConfig(outer_holdout=0.2),
    random_state=SEED,
)
t0 = time.perf_counter()
model.fit(X, y)
fit_seconds = time.perf_counter() - t0
print(f"fit took {fit_seconds:.1f}s")

INFO honestml.composition.presets: preset balanced applied: ['ensemble']


INFO honestml.adapters.reader: auto-typing column=education.num dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml: stage key=run stage=selection elapsed=128.1s


INFO honestml: stage key=run stage=ensemble elapsed=16.6s


WARNING honestml.adapters.boosting: boosting 'catboost' trained without early stopping; leaderboard comparison may favor overfit settings


INFO honestml: stage key=run stage=refit elapsed=13.4s


INFO honestml: stage key=run stage=refit elapsed=14.4s


INFO honestml: stage key=run stage=finalize elapsed=14.4s


fit took 172.8s


In [5]:
report = model.run_report_
print("preset:", report["preset"])
print("ensemble:", json.dumps(report["ensemble"], default=str))
pd.DataFrame(report["leaderboard"])

preset: {'name': 'balanced', 'applied': ['ensemble']}
ensemble: {"applied": false, "method": "caruana", "member_ids": ["baseline", "catboost", "lightgbm", "linear", "xgboost"], "weights": {"baseline": 0.5197889182058048, "catboost": 0.16358839050131926, "lightgbm": 0.10026385224274406, "linear": 0.158311345646438, "xgboost": 0.05804749340369393}, "gate_reason": "worse_than_best", "oof_delta": -0.005451251656861111}


,model_id,score,rank
0,catboost,0.928592,1
1,lightgbm,0.927600,2
2,xgboost,0.927052,3
3,linear,0.827823,4
4,baseline,0.499904,5


## The honesty check

In [6]:
selection = next(e["score"] for e in report["leaderboard"] if e["model_id"] == report["winner"])
print(f"winner          : {report['winner']}")
print(f"equivalence band: {report['band']['member_ids']}")
print(f"selection score : {selection:.4f}   (out-of-fold ROC AUC)")
print(f"holdout score   : {report['holdout_score']:.4f}   (untouched 20%, scored once)")
print(f"optimism        : {selection - report['holdout_score']:+.4f}")

winner          : catboost
equivalence band: ['catboost']
selection score : 0.9286   (out-of-fold ROC AUC)
holdout score   : 0.9235   (untouched 20%, scored once)
optimism        : +0.0051


In [7]:
save_run_report(report, RESULTS)
md_path = render_report(report, RESULTS, fmt="md")
Markdown(md_path.read_text(encoding="utf-8"))

# AutoML run report

## Run

| | |
|---|---|
| task | binary |
| metric | roc_auc |
| winner | catboost |
| holdout_score | 0.92348 |
| honestml_version | 1.0.0 |
| run_fingerprint | e6dfd0be2b4c8e4d8e9df968ef80fc9bb1c0541f30ebabff70daf56a8a9f9f5a |
| significance | bootstrap |
| preset | balanced (applied: ensemble) |

## Equivalence band

| | |
|---|---|
| members | catboost |
| width | 1 |
| unstable | False |
| winner_by_tiebreak | False |

## Budget

| | |
|---|---|
| mode | none |
| exhausted | False |
| exhausted_by | n/a |
| skipped | n/a |

## Ensemble

| | |
|---|---|
| applied | False |
| method | caruana |
| members | baseline, catboost, lightgbm, linear, xgboost |
| gate_reason | worse_than_best |
| oof_delta | -0.00545125 |

## Serving

| | |
|---|---|
| finalize | True |
| shipped_on | all |
| outer_holdout | 0.2 |

## Leaderboard

| rank | model | score |
|---|---|---|
| 1 | catboost (winner) | 0.928592 |
| 2 | lightgbm | 0.9276 |
| 3 | xgboost | 0.927052 |
| 4 | linear | 0.827823 |
| 5 | baseline | 0.499904 |

## Timings (s)

| stage | elapsed |
|---|---|
| run.selection | 128.1 |
| run.ensemble | 16.6 |
| run.refit | 14.4 |
| run.finalize | 14.4 |

## Resolved config

```json
{
  "seed": 42,
  "cv": {
    "scheme": "stratified",
    "n_splits": 5,
    "n_test": 1,
    "n_es": 1,
    "purge": 0,
    "embargo": 0,
    "period": null,
    "period_size": null,
    "step_periods": null,
    "purge_delta": null,
    "embargo_delta": null,
    "max_train_periods": null,
    "max_train_size": null,
    "weighting": "pooled",
    "calibrate": "off",
    "selection": "raw",
    "refinement_min_oof": 2000,
    "outer_holdout": 0.2
  },
  "budget": {
    "mode": "none",
    "time_budget_s": null,
    "n_trials": null,
    "memory_limit_mb": null
  },
  "fe": {
    "target_encoding": false,
    "te_smoothing": 10.0,
    "frequency_encoding": false,
    "intersections": false,
    "max_pairs": 50
  },
  "fs": null,
  "hpo": null,
  "ensemble": {
    "method": "caruana",
    "size": 50,
    "n_bags": 20,
    "metric": null,
    "random_state": 42
  },
  "significance": "bootstrap",
  "run_mode": "full",
  "model_types": [
    "catboost",
    "lightgbm"
  ]
}
```


## Comparison with published results

Published reference points on Adult (dataset versions and protocols differ,
and most of the literature reports accuracy-family metrics — read this as
context, not a controlled comparison):

| Source | Setup | Reported result |
| --- | --- | --- |
| UCI `adult.names` (Kohavi, 1996) | 16 classic algorithms, official test split | best error 14.05% (FSS Naive Bayes) ≈ 86% accuracy |
| [A Statistical Approach to Adult Census Income Level Prediction (arXiv:1810.10076)](https://arxiv.org/abs/1810.10076) | grid-searched Gradient Boosting | 88.16% accuracy |
| [Auto-sklearn 2.0 (arXiv:2007.04074)](https://arxiv.org/abs/2007.04074), Table 16 | AutoML, 60-minute budget, OpenML task 126025 | balanced error 0.154 ≈ 84.6% balanced accuracy |

What the cells above showed for honestml:

- **ROC AUC 0.92 on an untouched 20% holdout** after a ~3-minute fit with
  zero manual preprocessing beyond `replace("?", NA)`, and **optimism ≈
  +0.005** (selection 0.9286 vs holdout 0.9235): the number quoted is the
  kind you can ship;
- the `balanced` preset tried Caruana blending and the gate **refused to ship
  it** (`applied: false`, `gate_reason: worse_than_best`, OOF delta −0.005) —
  an ensemble has to *prove* it beats the single winner, and here it did not.
  An AutoML that always ships a blend would have shipped a worse model.

## Level 2: the full pipeline — HPO, feature engineering, feature selection

Level 1 used the `balanced` preset; now the heavy machinery:

- `preset="best"` + an explicit `HPOConfig(n_trials=30, timeout_s=600)` — an
  explicit argument always beats the preset's section, so HPO is capped at 30
  Optuna trials / 10 minutes per model (32k rows make the default 50 trials
  needlessly long); the preset still contributes gated ensembling;
- full FE for a binary task: cross-fit target encoding (anti-leakage,
  ADR-0041), frequency encoding, categorical intersections;
- `FeatureSelectionConfig(strategy="null_importance", cutoff="auto")` —
  features must beat their own target-shuffled null distribution to survive;
- the same untouched 20% outer holdout.

In [8]:
from honestml import FeatureSelectionConfig, FEConfig, HPOConfig

model_full = AutoML(
    task="binary",
    metric="roc_auc",
    preset="best",
    hpo=HPOConfig(n_trials=30, timeout_s=600),
    feature_engineering=FEConfig(target_encoding=True, frequency_encoding=True, intersections=True),
    feature_selection=FeatureSelectionConfig(strategy="null_importance", cutoff="auto"),
    cv=CVConfig(outer_holdout=0.2),
    random_state=SEED,
)
t0 = time.perf_counter()
model_full.fit(X, y)
print(f"level-2 fit took {(time.perf_counter() - t0) / 60:.1f} min")

INFO honestml.composition.presets: preset best applied: ['ensemble']


INFO honestml.adapters.reader: auto-typing column=education.num dtype=Int64 role=categorical reason=low_cardinality_int


WARNING honestml.composition.build: strategy 'null_importance' refits the ranker-model n_folds x (1 + n_runs=30) times


C:\Users\Admin\Documents\AutoML\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING honestml.adapters.boosting: boosting 'lightgbm' trained without early stopping; leaderboard comparison may favor overfit settings


...

INFO honestml: stage key=run stage=ensemble elapsed=17.2s


INFO honestml: stage key=run stage=refit elapsed=0.6s


INFO honestml: stage key=run stage=refit elapsed=0.6s


INFO honestml: stage key=run stage=finalize elapsed=0.6s


level-2 fit took 22.1 min


In [9]:
report_full = model_full.run_report_
sel_full = next(e["score"] for e in report_full["leaderboard"] if e["model_id"] == report_full["winner"])
print(f"winner          : {report_full['winner']}")
print(f"selection score : {sel_full:.4f}   (level 1: {selection:.4f})")
print(f"holdout score   : {report_full['holdout_score']:.4f}   (level 1: {report['holdout_score']:.4f})")
print(f"optimism        : {sel_full - report_full['holdout_score']:+.4f}")
print()
print("preset  :", report_full["preset"])
print("hpo     :", json.dumps(report_full["hpo"], default=str)[:600])
print("fs      :", json.dumps(report_full["feature_selection"], default=str)[:600])
print("ensemble:", json.dumps(report_full["ensemble"], default=str)[:300])
pd.DataFrame(report_full["leaderboard"])

winner          : lightgbm
selection score : 0.9184   (level 1: 0.9286)
holdout score   : 0.9141   (level 1: 0.9235)
optimism        : +0.0043

preset  : {'name': 'best', 'applied': ['ensemble']}
hpo     : {"backend": "optuna", "inner_cv": 3, "deterministic": false, "selection_oof_is_post_tuning": true, "tuned_on_full_feature_space": true, "cost_estimate_fits": 270, "tuned": {"catboost": {"chosen_params": {"depth": 6, "learning_rate": 0.04939461883492241, "iterations": 450, "l2_leaf_reg": 3.1976875222978762, "subsample": 0.7628596699871152, "one_hot_max_size": 49}, "inner_best_score": 0.9271813884198186, "n_trials_run": 27}, "lightgbm": {"chosen_params": {"max_depth": 9, "learning_rate": 0.043361369236984276, "n_estimators": 200, "reg_lambda": 0.9435411608572446, "subsample": 0.8406162739431632,
fs      : {"strategy": "null_importance", "n_selected": 27, "selected": ["capital.gain", "capital.loss", "education.num_freq", "marital.status_freq", "relationship_freq", "sex_freq", "education

,model_id,score,rank
0,lightgbm,0.918381,1
1,catboost,0.916734,2
2,xgboost,0.913540,3
3,linear,0.898074,4
4,baseline,0.499904,5


## Level 1 vs level 2: what the full pipeline buys

Level 2 comes out **slightly worse**, honestly measured: holdout AUC 0.9235 →
0.9141, selection 0.9286 → 0.9184, at ~22 min against ~3 min. The report
explains it: HPO's inner-CV best scores (~0.927) only match the untuned
defaults — the boosters' defaults are already near-optimal here — while the
feature-selection step keeps 27 of 68 post-FE features and trades away a
little signal; the ensemble gate again refuses to ship a blend that is not
better (`gate_reason: worse_than_best`). Feature selection
carries its own honest no-selection gate — it ships the reduced set only when
that subset is not worse than keeping everything — so here the 27-feature
subset passes the gate yet HPO+FE still cannot improve on the already-strong
defaults.

Verdict for this dataset: ship level 1. And that verdict needs no
leaderboard — both run reports state their honest holdout numbers.